# Task 5 — Convolutional Neural Networks
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
- Custom CNN: 3 Conv blocks (Conv+BN+ReLU+MP), 2 FC layers, Dropout
- Data augmentation (horizontal flip, random crop, color jitter)
- Learning rate scheduler (ReduceLROnPlateau)
- Transfer learning: ResNet-50 frozen backbone then partial fine-tuning
- Filter visualisation, activation maps, sample predictions
- Dataset: CIFAR-10 (10 classes, 60k images, 32x32x3)

## Step 0 — Imports & Setup

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, MobileNet
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix


OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')

# print(f'TensorFlow version: 2.21.0')  # set by builder
print('TF OK — GPU:', len(tf.config.list_physical_devices('GPU')) > 0)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

TF OK — GPU: False
GPU available: False


## Step 1 — Data Loading & Preprocessing (CIFAR-10)

In [2]:
def prepare_cifar10_data():
    """Load CIFAR-10; normalise to [0,1]; return train/val/test split."""
    (x_train_full, y_train_full), (x_test, y_test) = \
        keras.datasets.cifar10.load_data()
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test       = x_test.astype('float32') / 255.0
    x_val   = x_train_full[-5000:];  y_val   = y_train_full[-5000:]
    x_train = x_train_full[:-5000];  y_train = y_train_full[:-5000]
    y_train = to_categorical(y_train, 10)
    y_val   = to_categorical(y_val,   10)
    y_test  = to_categorical(y_test,  10)
    print('Using CIFAR-10 dataset')
    print(f'Train: {x_train.shape}  Val: {x_val.shape}  Test: {x_test.shape}')
    print(f'Classes: {len(np.unique(np.argmax(y_train, axis=1)))}')
    return x_train, x_val, x_test, y_train, y_val, y_test

x_train, x_val, x_test, y_train, y_val, y_test = prepare_cifar10_data()
INPUT_SHAPE = x_train.shape[1:]   # (32, 32, 3)
NUM_CLASSES = y_train.shape[1]    # 10

Using CIFAR-10 dataset
Train: (45000, 32, 32, 3)  Val: (5000, 32, 32, 3)  Test: (10000, 32, 32, 3)
Classes: 10


## Step 2 — Custom CNN Architecture

```
Input(32x32x3) -> Conv(32,3x3) BN ReLU MaxPool(2x2)
             -> Conv(64,3x3) BN ReLU MaxPool(2x2)
             -> Conv(128,3x3) BN ReLU MaxPool(2x2)
             -> Flatten -> Dense(256) BN ReLU Dropout(0.5)
             -> Dense(128) BN ReLU Dropout(0.3)
             -> Dense(10, Softmax)
```

In [4]:
def build_custom_cnn(input_shape, num_classes):
    """3 Conv blocks + 2 FC layers; BN + Dropout."""
    return models.Sequential([
        layers.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Block 3
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Head
        layers.Flatten(),
        layers.Dense(256), layers.BatchNormalization(),
        layers.Activation('relu'), layers.Dropout(0.5),
        layers.Dense(128), layers.BatchNormalization(),
        layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ], name='custom_cnn')
print('Building custom CNN...')
model_custom = build_custom_cnn(INPUT_SHAPE, NUM_CLASSES)
model_custom.summary()

Building custom CNN...


Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 654,410 (2.50 MB)

 Trainable params: 653,194 (2.49 MB)

 Non-trainable params: 1,216 (4.75 KB)

## Step 3 — Training with Data Augmentation

In [5]:
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1)
datagen.fit(x_train)
lr_sched   = callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, verbose=1)
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)
model_custom.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                    loss='categorical_crossentropy', metrics=['accuracy'])
print('Training custom CNN (CIFAR-10, 30 epochs, augmented)...')
history_custom = model_custom.fit(
    datagen.flow(x_train, y_train, batch_size=128),
    validation_data=(x_val, y_val),
    epochs=30, callbacks=[lr_sched, early_stop], verbose=2)

Training custom CNN (CIFAR-10, 30 epochs, augmented)...
Epoch 1/30
352/352 - 45s - 129ms/step - accuracy: 0.4172 - loss: 1.6247 - val_accuracy: 0.2618 - val_loss: 2.3313 - learning_rate: 0.0010
Epoch 2/30
352/352 - 41s - 116ms/step - accuracy: 0.5572 - loss: 1.2389 - val_accuracy: 0.6168 - val_loss: 1.0810 - learning_rate: 0.0010
Epoch 3/30
352/352 - 37s - 106ms/step - accuracy: 0.6092 - loss: 1.1092 - val_accuracy: 0.5334 - val_loss: 1.5000 - learning_rate: 0.0010
Epoch 4/30
352/352 - 37s - 104ms/step - accuracy: 0.6417 - loss: 1.0161 - val_accuracy: 0.5612 - val_loss: 1.2136 - learning_rate: 0.0010
Epoch 5/30
352/352 - 38s - 109ms/step - accuracy: 0.6666 - loss: 0.9517 - val_accuracy: 0.6916 - val_loss: 0.9423 - learning_rate: 0.0010
Epoch 6/30
352/352 - 41s - 116ms/step - accuracy: 0.6823 - loss: 0.9074 - val_accuracy: 0.6786 - val_loss: 0.9367 - learning_rate: 0.0010
Epoch 7/30
352/352 - 40s - 114ms/step - accuracy: 0.6967 - loss: 0.8661 - val_accuracy: 0.6340 - val_loss: 1.1061 - 

## Step 4 — Training History (Accuracy + Loss)

In [8]:
ep = range(1, len(history_custom.history['accuracy'])+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, history_custom.history['accuracy'],     'b-', label='Train')
axes[0].plot(ep, history_custom.history['val_accuracy'], 'g--',label='Val')
axes[0].set_title('Custom CNN — Accuracy'); axes[0].legend()
axes[1].plot(ep, history_custom.history['loss'],     'b-', label='Train')
axes[1].plot(ep, history_custom.history['val_loss'], 'g--', label='Val')
axes[1].set_title('Custom CNN — Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_training_history.png'), dpi=150)
plt.close()
print('Saved T5_training_history.png')

Saved T5_training_history.png


## Step 5 — Custom CNN Evaluation & Confusion Matrix

In [10]:
test_loss_c, test_acc_c = model_custom.evaluate(x_test, y_test, verbose=0)
print(f'Custom CNN Test — Acc: {test_acc_c:.4f}  Loss: {test_loss_c:.4f}')
y_pred_c = np.argmax(model_custom.predict(x_test, verbose=0), axis=1)
y_true   = np.argmax(y_test, axis=1)
cm_c     = confusion_matrix(y_true, y_pred_c)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Custom CNN — Confusion Matrix  (Acc: {test_acc_c:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_custom_confusion_matrix.png'), dpi=150)
plt.close()
print('Saved T5_custom_confusion_matrix.png')
print(classification_report(y_true, y_pred_c, target_names=CLASS_NAMES))

Custom CNN Test — Acc: 0.8241  Loss: 0.5032


ValueError: Found input variables with inconsistent numbers of samples: [100000, 10000, 100000]

## Step 6 — ResNet-50 Transfer Learning (Frozen Backbone)

In [11]:
base_rn = ResNet50(weights='imagenet', include_top=False, input_shape=(32,32,3))
base_rn.trainable = False
model_rn = models.Sequential([
    keras.Input(shape=(32,32,3)),
    base_rn,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), layers.BatchNormalization(),
    layers.Dense(10, activation='softmax'),
], name='resnet50_transfer')
model_rn.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
model_rn.summary()

Model: "resnet50_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,115,850 (91.99 MB)

 Trainable params: 527,626 (2.01 MB)

 Non-trainable params: 23,588,224 (89.98 MB)

## Step 7 — ResNet-50 Training & History

In [ ]:
# Phase 1 — frozen backbone, head training (5 epochs)
print('Phase 1 — training head only (5 epochs)...')
base_rn.trainable = False
phase1 = model_rn.fit(
    x_train, y_train, batch_size=64,
    validation_data=(x_val, y_val), epochs=5, verbose=2)

# Phase 2 — partial fine-tuning, top 30 layers unfrozen (3 epochs, lr=1e-5)
print('Phase 2 — unfreezing top 30 layers, lr=1e-5 (3 epochs)...')
base_rn.trainable = True
for layer in base_rn.layers[:-30]:
    layer.trainable = False
model_rn.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])
phase2 = model_rn.fit(
    x_train, y_train, batch_size=64,
    validation_data=(x_val, y_val), epochs=3, verbose=2)

# Plot combined history
h  = phase1.history
h2 = phase2.history
ep1 = range(1, len(h['accuracy'])+1)
ep2 = range(len(h['accuracy'])+1, len(h['accuracy'])+len(h2['accuracy'])+1)
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
for ax, key in zip(ax2, ['accuracy', 'loss']):
    ax.plot(ep1, h[key],       'r-o', label='P1-Train')
    ax.plot(ep1, h['val_'+key], 'r--', label='P1-Val')
    ax.plot(ep2, h2[key],       'b-o', label='P2-Train')
    ax.plot(ep2, h2['val_'+key], 'b--', label='P2-Val')
    ax.set_title(f'ResNet-50 Transfer — {key.title()}'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_training_history.png'), dpi=150)
plt.close()
print('Saved T5_resnet50_training_history.png')

Phase 1 — training head only (5 epochs)...
Epoch 1/5
704/704 - 52s - 73ms/step - accuracy: 0.1481 - loss: 2.2601 - val_accuracy: 0.0996 - val_loss: 2.3265
Epoch 2/5


KeyboardInterrupt: 

## Step 8 — ResNet-50 Evaluation & Confusion Matrix

In [ ]:
test_loss_r, test_acc_r = model_rn.evaluate(x_test, y_test, verbose=0)
print(f'ResNet-50 Test — Acc: {test_acc_r:.4f}  Loss: {test_loss_r:.4f}')
y_pred_r = np.argmax(model_rn.predict(x_test, verbose=0), axis=1)
cm_r     = confusion_matrix(y_true, y_pred_r)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_r, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'ResNet-50 Transfer Learning  (Acc: {test_acc_r:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_confusion_matrix.png'), dpi=150)
plt.close()
print('Saved T5_resnet50_confusion_matrix.png')
print(classification_report(y_true, y_pred_r, target_names=CLASS_NAMES))
with open(os.path.join(OUTPUT_DIR, 'resnet50_metrics.txt'), 'w') as _fh:
    _fh.write(f'{test_acc_r:.4f}\n{test_loss_r:.4f}\n')

## Step 9 — Learned Filter Visualisation

In [ ]:
# First conv layer filters (R channel, 32 filters)
first_conv = model_custom.layers[0]
w  = first_conv.get_weights()[0]   # (3,3,3,32)
fig, axes = plt.subplots(4, 8, figsize=(22, 8))
axes = axes.flatten()
for i in range(32):
    f = w[:,:,0,i]
    f_norm = (f - f.min()) / (f.max() - f.min() + 1e-8)
    axes[i].imshow(f_norm, cmap='viridis'); axes[i].axis('off')
    axes[i].set_title(f'#{i}', fontsize=6)
plt.suptitle('Custom CNN — First Conv Layer Learned Filters (32/32)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_filters.png'), dpi=150)
plt.close()
print('Saved T5_cnn_filters.png')

## Step 10 — Activation Map Visualisation

In [ ]:
extractor = keras.Model(
    inputs=model_custom.inputs[0],
    outputs=[l.output for l in model_custom.layers
             if isinstance(l, (layers.Conv2D, layers.Activation))])
acts = extractor.predict(x_test[:3], verbose=0)
b1   = acts[4]    # Conv Block 1: (3, 16, 16, 32)
fig, axes = plt.subplots(3, 16, figsize=(24, 6))
for si in range(3):
    for ch in range(min(16, b1.shape[-1])):
        axes[si,ch].imshow(b1[si,:,:,ch], cmap='viridis')
        axes[si,ch].axis('off')
plt.suptitle('CNN Activation Maps — Conv Block 1 (3 images x 16 channels)', 
             fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_activations.png'), dpi=150)
plt.close()
print('Saved T5_cnn_activations.png')

## Step 11 — Sample Image Predictions

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(x_test), 16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i, j in enumerate(idx):
    ax = axes.flatten()[i]
    ax.imshow(x_test[j])
    pred_l = CLASS_NAMES[y_pred_c[j]]
    true_l = CLASS_NAMES[y_true[j]]
    color  = 'green' if y_pred_c[j] == y_true[j] else 'red'
    ax.set_title(f'CNN: {pred_l}\nTrue: {true_l}', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle(
    'Custom CNN — CIFAR-10 Sample Predictions (green=correct, red=wrong)',
    fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_sample_predictions.png'), dpi=150)
plt.close()
print('Saved T5_sample_predictions.png')

## Step 12 — Model Comparison: Custom CNN vs ResNet-50

In [ ]:
models_list  = ['Custom CNN', 'ResNet-50\n(Transfer Learning)']
test_accs    = [test_acc_c, test_acc_r]
test_losses  = [test_loss_c, test_loss_r]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, vals, tit in zip(axes,
                          [test_accs, test_losses],
                          ['Test Accuracy', 'Test Loss']):
    b = ax.bar(models_list, vals,
               color=['#4472C4','#ED7D31'], edgecolor='black', width=0.5)
    ax.set_title(tit); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(b, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
               f'{val:.4f}', ha='center', fontweight='bold')
plt.suptitle('Task 5 — CNN Model Comparison on CIFAR-10',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_model_comparison.png'), dpi=150)
plt.close()
print(f'Custom CNN: acc={test_acc_c:.4f}  loss={test_loss_c:.4f}')
print(f'ResNet-50 : acc={test_acc_r:.4f}  loss={test_loss_r:.4f}')
print('Saved T5_model_comparison.png')

## Step 13 — Results Discussion

In [ ]:
print('\n=== T5 RESULTS SUMMARY ===')
print(f'Custom CNN (from scratch): acc={test_acc_c:.4f}')
print(f'ResNet-50 (transfer):      acc={test_acc_r:.4f}')
print('')\n
print('Key take-aways:')
print('- Augmentation: rot, flip, shift, zoom — active in training.')
print('- Filters show edge / texture detectors.')
print('- Activation maps highlight channel-specific feature detection.')
print('- ResNet-50 needs full fine-tuning >>30 epochs at lr=1e-5 on 32x32.')
print('- Next: EfficientNet-B0, TTA, warm-up + cosine schedule.')